In [235]:
import pandas as pd
import numpy as np

In [236]:
# import os
# print(os.getcwd())
df = pd.read_excel('../data/market_data_raw.xlsx')
#inspecting the dataframe
df.head()

,date,ticker,sector,asset_class,price,volume,analyst
0,2024-04-11,JPM,Financials,Equity,153.25,6734228.0,GS
1,2024-04-18,AAPL,Tech,Equity,143.02,1660251.0,JPM
2,2024-06-06,XOM,Energy,Equity,141.17,3720515.0,MS
3,2024-02-12,JPM,Financials,Equity,144.48,4880541.0,NaN
4,2024-05-20,JPM,Financials,Equity,158.28,2889015.0,NaN


In [237]:
df.shape

(654, 7)

In [238]:
df.dtypes

date            object
ticker          object
sector          object
asset_class     object
price          float64
volume         float64
analyst         object
dtype: object

In [239]:
df.columns

Index(['date', 'ticker', 'sector', 'asset_class', 'price', 'volume',
       'analyst'],
      dtype='object')

In [240]:
df_clean = df.copy()

In [241]:
df_clean['date'] = pd.to_datetime(df['date'], format='%Y-%m-%d')
df_clean.dtypes

date           datetime64[ns]
ticker                 object
sector                 object
asset_class            object
price                 float64
volume                float64
analyst                object
dtype: object

In [242]:
#checking missing values in each column
total = df.isnull().sum()
total

date            0
ticker          0
sector          0
asset_class     0
price          21
volume          5
analyst        54
dtype: int64

In [243]:
#checking % of missing values in each column
df_clean.isnull().mean() * 100

date           0.000000
ticker         0.000000
sector         0.000000
asset_class    0.000000
price          3.211009
volume         0.764526
analyst        8.256881
dtype: float64

In [244]:
#standardizing the ticker and asset_class columns to uppercase and removing leading/trailing spaces
df_clean[df_clean['price'].isnull()]['ticker'].value_counts()
df_clean['ticker'] = df_clean['ticker'].str.upper()
df_clean['ticker'] = df_clean['ticker'].str.strip()
df_clean['asset_class'].value_counts()
df_clean['asset_class'] = df_clean['asset_class'].str.upper()
df_clean['asset_class'] = df_clean['asset_class'].str.strip()



In [245]:

# df_clean[df_clean['price'].isnull() & df_clean['volume'].isnull()]
# df_clean['price'] = df_clean.groupby('ticker')['price'].ffill()
# df_clean['volume'] = df_clean.groupby('ticker')['volume'].ffill()


In [246]:
#checking for duplicates
df_clean.duplicated().sum()
df_clean[df_clean.duplicated(['ticker', 'date'],keep = False)].sort_values(by=['ticker', 'date'])

,date,ticker,sector,asset_class,price,volume,analyst
140,2024-02-19,MSFT,Tech,EQUITY,NaN,5748297.0,MS
413,2024-02-19,MSFT,Tech,EQUITY,NaN,5748297.0,MS
76,2024-05-27,MSFT,Tech,EQUITY,288.020000,6694943.0,GS
582,2024-05-27,MSFT,Tech,EQUITY,288.302810,6694943.0,GS
5,2024-05-30,MSFT,Tech,EQUITY,296.901992,8341767.0,JPM
245,2024-05-30,MSFT,Tech,EQUITY,297.170000,8341767.0,JPM
270,2024-06-21,MSFT,Tech,EQUITY,296.468386,8154033.0,GS
453,2024-06-21,MSFT,Tech,EQUITY,296.670000,8154033.0,GS


In [247]:
#handling duplicates and negative volume values
df_clean = df_clean.drop_duplicates(subset=['ticker', 'date'], keep='first')
df_clean.loc[df_clean['volume'] < 0, 'volume'] = df_clean.loc[df_clean['volume'] < 0, 'volume'].abs()



In [248]:
#forward fill the missing values for price and volume based on ticker
df_clean['price'] = df_clean.groupby('ticker')['price'].ffill()
df_clean['volume'] = df_clean.groupby('ticker')['volume'].ffill()
df_clean['analyst'] = df_clean.groupby('ticker')['analyst'].fillna('Unknown')
df_clean.isnull().sum()


date           0
ticker         0
sector         0
asset_class    0
price          0
volume         0
analyst        0
dtype: int64

In [249]:
#z-score method to identify outliers in the price column for each ticker. A z-score greater than 3 or less than -3 is considered an outlier.
#based on the z-score we can decide to either remove the outlier, clip it or interpolate it. In this case we will clip the outliers to the upper and lower bounds of the z-score.
df_clean['ticker_mean'] = df_clean.groupby('ticker')['price'].transform('mean')
df_clean['ticker_std'] = df_clean.groupby('ticker')['price'].transform('std')
df_clean['z_score'] = (df_clean['price'] - df_clean['ticker_mean']) / df_clean['ticker_std']
df_clean['is_outlier'] = df_clean['z_score'].abs() > 3
lower_bound = df_clean['ticker_mean'] - 3*df_clean['ticker_std']
upper_count = df_clean['ticker_mean'] + 3*df_clean['ticker_std']
df_clean[df_clean['is_outlier'] == True]
df_clean['price'] = df_clean['price'].clip(lower = lower_bound, upper = upper_count)
#clipping dint work as expected so we need to interpolate the outliers (1 value before the outlier and 1 value after the outlier) to get a better estimate of the outlier value.
df_clean.loc[121, 'price'] = (df_clean.loc[409, 'price'] + df_clean.loc[255, 'price']) / 2
df_clean[(df_clean['date'] > '2024-03-20') &  (df_clean['date'] < '2024-03-30') & (df_clean['ticker'] == 'AAPL')].sort_values(by = ['date'])


,date,ticker,sector,asset_class,price,volume,analyst,ticker_mean,ticker_std,z_score,is_outlier
623,2024-03-21,AAPL,Tech,EQUITY,141.490,8765637.0,MS,166.707308,214.001332,-0.117837,False
409,2024-03-22,AAPL,Tech,EQUITY,144.580,8062055.0,JPM,166.707308,214.001332,-0.103398,False
121,2024-03-25,AAPL,Tech,EQUITY,141.675,5916151.0,JPM,166.707308,214.001332,11.304475,True
255,2024-03-26,AAPL,Tech,EQUITY,138.770,8836357.0,MS,166.707308,214.001332,-0.130547,False
563,2024-03-27,AAPL,Tech,EQUITY,138.570,6454149.0,Unknown,166.707308,214.001332,-0.131482,False
14,2024-03-28,AAPL,Tech,EQUITY,142.850,801862.0,Unknown,166.707308,214.001332,-0.111482,False
568,2024-03-29,AAPL,Tech,EQUITY,141.900,4694751.0,GS,166.707308,214.001332,-0.115921,False


In [250]:
#checking for stale records where the price is the same for consecutive days for a given ticker. This could be an indication of stale data or a lack of trading activity for that ticker.
df_clean = df_clean.sort_values(['ticker', 'date']).reset_index(drop=True)
df_clean['price_diff'] = df_clean.groupby('ticker')['price'].diff()
df_clean[df_clean['price_diff'] == 0].sort_values(by=['ticker', 'date'])



,date,ticker,sector,asset_class,price,volume,analyst,ticker_mean,ticker_std,z_score,is_outlier,price_diff
319,2024-03-22,MSFT,Tech,EQUITY,296.60,5035574.0,GS,294.808772,9.471891,0.189110,False,0.0
490,2024-05-20,T10Y,Rates,FIXED INCOME,98.53,7535463.0,Unknown,101.611154,4.516117,-0.682257,False,0.0
491,2024-05-21,T10Y,Rates,FIXED INCOME,98.53,8120839.0,JPM,101.611154,4.516117,-0.682257,False,0.0
492,2024-05-22,T10Y,Rates,FIXED INCOME,98.53,5627640.0,MS,101.611154,4.516117,-0.682257,False,0.0
493,2024-05-23,T10Y,Rates,FIXED INCOME,98.53,4558456.0,JPM,101.611154,4.516117,-0.682257,False,0.0


In [251]:

for ticker in df_clean['ticker'].unique():
    all_dates = pd.date_range(start = df_clean['date'].min(), end= df_clean['date'].max(),freq = 'B')
    ticker_dates = df_clean[df_clean['ticker'] == ticker]['date']
    missing_dates = all_dates.difference(ticker_dates)
    print(f'Ticker: {ticker}, Missing Dates: {missing_dates}')
df_clean['ticker'].value_counts()

Ticker: AAPL, Missing Dates: DatetimeIndex([], dtype='datetime64[ns]', freq='B')
Ticker: JPM, Missing Dates: DatetimeIndex([], dtype='datetime64[ns]', freq='B')
Ticker: MSFT, Missing Dates: DatetimeIndex([], dtype='datetime64[ns]', freq='B')
Ticker: T10Y, Missing Dates: DatetimeIndex([], dtype='datetime64[ns]', freq='B')
Ticker: XOM, Missing Dates: DatetimeIndex([], dtype='datetime64[ns]', freq='B')


ticker
AAPL    130
JPM     130
MSFT    130
T10Y    130
XOM     130
Name: count, dtype: int64